In [1]:
%reset -f

import os
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from PIL import Image
import torchvision
from torchvision.transforms.functional import InterpolationMode
from torchvision.io.image import ImageReadMode
from tqdm import tqdm
import json
from torch.utils.data import Dataset, DataLoader, random_split
import gc
from typing import Any
from transformers import AutoModel
from diffusers import AutoencoderKL, FluxTransformer2DModel, BitsAndBytesConfig
from pathlib import Path
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

torch.cuda.empty_cache()

c:\Users\Tartiflettopolys\Desktop\im_a_folder\Perso\Prog\Python\Pokémon Fine Tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0506 00:45:27.554000 19820 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
def load_env():
    with open("./config.json", "r") as f:
        config: dict[str, Any] = json.loads(f.read())
        if "env" in config:
            for key, value in config["env"].items():
                os.environ[key] = value

load_env()

login(os.environ.get("HF_TOKEN", ""))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
class PokemonDataset(Dataset):
    def __init__(self, data: torch.Tensor, captions: list[str]):
        assert data.shape[0] == len(captions)
        self.data:torch.Tensor = data
        self.captions = captions

    def __len__(self) -> int:
        return self.data.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, str]:
        return (self.data[idx], self.captions[idx])

In [4]:
def resize_dataset(raw_dir: str = "./images/raw", target_dir: str = "./images/1024x1024") -> None:
    if os.path.exists(target_dir):
        os.rmdir(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    valid_extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
    files = [f for f in os.listdir(raw_dir) if f.lower().endswith(valid_extensions)]

    trans_resize = torchvision.transforms.Resize((1024, 1024), InterpolationMode.BICUBIC, antialias=True)

    for filename in tqdm(files, desc="Resizing images"):
        img_path: str = os.path.join(raw_dir, filename)
        raw_image = torchvision.io.read_image(img_path, ImageReadMode.RGB_ALPHA)
        
        alpha = raw_image[3:4, :, :].float() / 255.0
        rgb = raw_image[:3, :, :].float()
        white_bg = torch.ones_like(rgb) * 255.0
        
        flat_image = (rgb * alpha + white_bg * (1.0 - alpha)).to(torch.uint8)

        output_image: torch.Tensor = trans_resize(flat_image).cpu() 
        torchvision.io.write_png(output_image, os.path.join(target_dir, filename), 0)

def load_dataset() -> PokemonDataset:
    images_dir:str = os.path.join(os.getcwd(), "images", "1024x1024")
    images_path:list[str] = os.listdir(images_dir)
    images:list[torch.Tensor] = list[torch.Tensor]()
    captions: list[str] = []

    captions_dict: dict[str, str] = {}
    with open("./captions.json", encoding="utf-8") as f:
        captions_dict = json.loads(f.read())

    for image_local_path in images_path:
        img_name: str = image_local_path.removesuffix(".png")
        if img_name not in captions_dict:
            print(f"{img_name} doesn't have caption!")
            continue

        image_absolute_path:str = os.path.join(images_dir, image_local_path)
        image:torch.Tensor = torchvision.io.read_image(image_absolute_path, ImageReadMode.RGB).type(torch.float32) / 255.0
        image = image * 2.0 - 1.0
        images.append(image)
        captions.append(captions_dict[img_name])

    return PokemonDataset(torch.stack(images).cpu(), captions)

def test_dataset() -> None:
    images = load_dataset()
    idx = random.randint(0, len(images) - 1)
    img_tensor, _ = images[idx]

    img_to_show = (img_tensor * 0.5 + 0.5).permute(1, 2, 0).numpy()

    plt.imshow(img_to_show)
    plt.title(f"Random Pokémon Sample (Index: {idx})")
    plt.axis("off")
    plt.show()

In [5]:
dataset: PokemonDataset = load_dataset()

In [ ]:
def train_pokemon_flux_lora(
    dataset: PokemonDataset,
    text_encoder_dir: str = "./models/text_encoder",
    transformer_path: str = "./models/diffusion_model",
    vae_path: str = "./models/vae",
    batch_size: int = 4,
    epochs: int = 20,
    learning_rate: float = 1e-4,
    device: str = "cuda",
    lora_rank: int = 16, # Rank of the LoRA update matrices
    lora_alpha: int = 32, # Scaling factor
    train_proportion: float = 0.9
) -> None:
    
    print("Loading VAE")
    vae = AutoencoderKL.from_pretrained(
        vae_path, 
        torch_dtype=torch.bfloat16
    ).to(device)
    vae.requires_grad_(False)

    print("Loading Text Encoder")
    text_encoder = AutoModel.from_pretrained(
        text_encoder_dir,
        dtype=torch.bfloat16,
    )
    text_encoder.requires_grad_(False)

    print("Loading FLUX.2 Transformer")
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        # Optional: keep certain sensitive layers in bfloat16 for stability
        llm_int8_skip_modules=["norm_q", "norm_k", "norm_added_k", "norm_added_q"] 
    )

    transformer = FluxTransformer2DModel.from_pretrained(
        transformer_path,
        subfolder="transformer",
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
        device_map={"": device} # <-- ADD THIS LINE
    )

    # from peft import prepare_model_for_kbit_training
    # transformer = prepare_model_for_kbit_training(transformer)
    transformer.requires_grad_(False)

    print("Injecting LoRA adapters into Transformer")
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        target_modules=[
            "to_q", "to_k", "to_v", "to_out.0", # Attention layers
            "proj_in", "proj_out", "ff.net.0.proj", "ff.net.2" # Feed-forward layers
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="FEATURE_EXTRACTION"
    )
    
    # Wrap the transformer with PEFT to add the trainable LoRA weights
    transformer = get_peft_model(transformer, lora_config)
    
    # # Enable gradient checkpointing to save VRAM during the backward pass
    # transformer.enable_gradient_checkpointing()
    
    transformer.print_trainable_parameters()

    print("Initializing Optimizer")
    optimizer = torch.optim.AdamW(
        transformer.parameters(),
        lr=learning_rate,
        weight_decay=1e-2,
        betas=(0.9, 0.999)
    )

    torch.cuda.empty_cache()

    print("Preparing dataset and dataloader")
    train_size: int = int(train_proportion * len(dataset))
    val_size: int = len(dataset) - train_size
    train_subset, _ = random_split(dataset, [train_size, val_size])
    dataloader: DataLoader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)

    return

    print("Starting Training Loop...")
    global_step: int = 0
    
    for epoch in range(epochs):
        epoch_loss: float = 0.0

        for step, (batch_images, batch_captions) in tqdm(enumerate(dataloader), desc=f"Epoch: {epoch + 1}"):
            optimizer.zero_grad()
            
            bsz: int = batch_images.shape[0]
            batch_images = batch_images.to(torch_device, dtype=weight_dtype)
            
            # --- A. Encode Images to Latents ---
            vae_scaling_factor: float = 0.18215 
            with torch.no_grad():
                # Sample latents from the diagonal Gaussian distribution of the VAE encoder
                latents: torch.Tensor = vae.encode(batch_images).latent_dist.sample()
                latents = latents * vae_scaling_factor
            
            # --- B. Process Text Captions ---
            text_inputs: dict[str, torch.Tensor] = tokenizer(
                list(batch_captions),  # Ensure it is a list of strings
                padding="max_length",
                max_length=256,
                truncation=True,
                return_tensors="pt"
            ).to(torch_device)
            
            with torch.no_grad():
                # Grab Qwen hidden states
                encoder_outputs: Any = text_encoder(**text_inputs, output_hidden_states=True)
                # Pull the penultimate layer for stable embeddings
                encoder_hidden_states: torch.Tensor = encoder_outputs.hidden_states[-2] 
                # Create the global pooled projection vector (cls/first token representation)
                pooled_projections: torch.Tensor = encoder_hidden_states[:, 0, :] 
            
            # --- C. Flow Matching Interpolation ---
            noise: torch.Tensor = torch.randn_like(latents)
            timesteps: torch.Tensor = torch.rand((bsz,), device=torch_device, dtype=weight_dtype)
            t_expand: torch.Tensor = timesteps.view(bsz, 1, 1, 1)
            
            # Linear interpolation: x_t = (1 - t) * x_0 + t * x_1
            x_t: torch.Tensor = (1.0 - t_expand) * noise + t_expand * latents
            
            # The flow matching target velocity vector v
            target_velocity: torch.Tensor = latents - noise
            
            # --- D. Forward Pass ---
            model_pred: torch.Tensor = transformer(
                hidden_states=x_t,
                timestep=timesteps, 
                encoder_hidden_states=encoder_hidden_states,
                pooled_projections=pooled_projections
            ).sample
            
            # --- E. MSE Velocity Loss & Optimization ---
            loss: torch.Tensor = nn.functional.mse_loss(model_pred, target_velocity, reduction="mean")
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            global_step += 1
            
            if global_step % 10 == 0:
                print(f"Step {global_step} | Loss: {loss.item():.4f}")
                
    print("Training complete! Saving LoRA weights...")
    transformer.save_pretrained("./pokemon_flux2_klein_lora")

train_pokemon_flux_lora(dataset, device="cpu", batch_size=1)

Loading VAE


Some weights of the model checkpoint at ./models/vae were not used when initializing AutoencoderKL: 
 ['bn.running_mean, bn.num_batches_tracked, bn.running_var']


Loading Text Encoder


Loading weights: 100%|██████████| 650/650 [00:00<00:00, 6786.98it/s]
[transformers] Qwen3Model LOAD REPORT from: ./models/text_encoder
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FLUX.2 Transformer


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 